In [1]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.5.1+cu121
CUDA Available: True
Device Name: NVIDIA GeForce RTX 3050


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

In [ ]:
# --- 1. 設定セクション ---
# ジャンルマップを定義
genres_map = {
    0:    '未設定',
    101:  '異世界（恋愛）', 102:  '現実世界（恋愛）',
    201:  'ハイファンタジー', 202:  'ローファンタジー',
    301:  '純文学', 302:  'ヒューマンドラマ', 303:  '歴史',
    304:  '推理', 305:  'ホラー', 306:  'アクション', 307:  'コメディー',
    401:  'VRゲーム', 402:  '宇宙', 403:  '空想科学', 404:  'パニック',
    9901: '童話', 9902: '詩', 9903: 'エッセイ', 9904: 'リプレイ',
    9999: 'その他', 9801: 'ノンジャンル'
}

TARGET_GENRE = None
NUM_SAMPLES_PER_CLASS = 10000

In [4]:
# --- 2. データの準備 ---
corpus_path = 'dataset/narou_dataset.csv' 
try:
    df_full = pd.read_csv(corpus_path)
except FileNotFoundError:
    print(f"エラー: ファイルが見つかりません - {corpus_path}")
    exit()

# 2-1. ジャンルによるデータ絞り込み
if TARGET_GENRE is not None:
    if TARGET_GENRE in genres_map:
        genre_name = genres_map[TARGET_GENRE]
        print(f"--- ジャンル: {genre_name} ({TARGET_GENRE}) を対象とします ---")
        df_base = df_full[df_full['作品ジャンル'] == TARGET_GENRE].copy()
        if len(df_base) == 0:
            print("エラー: 指定されたジャンルのデータが存在しません。")
            exit()
    else:
        print(f"エラー: 指定されたジャンルコード ({TARGET_GENRE}) が見つかりません。")
        exit()
else:
    print("--- 全てのジャンルを対象とします ---")
    df_base = df_full

# 2-2. サンプリング処理
if NUM_SAMPLES_PER_CLASS > 0:
    print(f"ランダムサンプリングを実行します (基本件数: {NUM_SAMPLES_PER_CLASS}件/クラス)。")
    df_eternal_0 = df_base[df_base['is_eternal'] == 0]
    df_eternal_1 = df_base[df_base['is_eternal'] == 1]
    
    n_samples = min(NUM_SAMPLES_PER_CLASS, len(df_eternal_0), len(df_eternal_1))
    
    if n_samples == 0:
        print("エラー: サンプリング可能なデータがありません（どちらかのクラスのデータが0件です）。")
        exit()

    print(f"両クラスの作品数が {n_samples}件 になるように調整して抽出します。")
    
    sampled_0 = df_eternal_0.sample(n=n_samples, random_state=42)
    sampled_1 = df_eternal_1.sample(n=n_samples, random_state=42)
    
    df = pd.concat([sampled_0, sampled_1])
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"サンプリング後の合計データ数: {len(df)}件")
else:
    print("全データを使用して学習します。")
    df = df_base

# 2-3. テキストの準備
# NaN値を空文字で埋める
# df['タイトル'] = df['タイトル'].fillna('')
# df['あらすじ'] = df['あらすじ'].fillna('')

# 学習に使用するテキストデータを結合
# print("タイトル、あらすじを結合しています...")
# df['text'] = df['タイトル'] + '。' + df['あらすじ']
df['text'] = df['あらすじ']

# ラベルとテキストを抽出
df = df[['text', 'is_eternal']].rename(columns={'is_eternal': 'labels'})

# 訓練データと検証データに分割 (80% 訓練, 20% 検証)
train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['labels'])

--- ジャンル: エッセイ (9903) を対象とします ---
ランダムサンプリングを実行します (基本件数: 5000件/クラス)。
両クラスの作品数が 2640件 になるように調整して抽出します。
サンプリング後の合計データ数: 5280件


In [5]:
# --- 3. モデルとトークナイザの準備 ---

# v3のLargeモデルを使用（精度重視）
MODEL_NAME = "tohoku-nlp/bert-large-japanese-v2"

# ターゲット設定
NUM_LABELS = 2

print(f"使用モデル: {MODEL_NAME}")

# トークナイザのロード
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# モデルのロード
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

# GPUへ転送
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("モデルとトークナイザの読み込み完了。")

使用モデル: tohoku-nlp/bert-large-japanese-v2


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at tohoku-nlp/bert-large-japanese-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


モデルとトークナイザの読み込み完了。


In [6]:
# --- 4. データセットのトークン化 ---
# Hugging FaceのDataset形式に変換
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

def tokenize_function(examples):
    # テキストをトークン化し、最大長512にパディング/切り詰め
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

print("データセットをトークン化しています...")
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

Parameter 'function'=<function tokenize_function at 0x00000226AE6D2B60> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


データセットをトークン化しています...


Map:   0%|          | 0/4224 [00:00<?, ? examples/s]

Map:   0%|          | 0/1056 [00:00<?, ? examples/s]

In [7]:
# --- 5. 評価指標の定義 ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [8]:
# --- 6. 学習の設定 ---
from transformers import TrainingArguments

# 出力ディレクトリ
output_dir = "./result_"+genres_map[TARGET_GENRE] if TARGET_GENRE is not None else "./result_all"

# 設定
training_args = TrainingArguments(
    output_dir=output_dir,
    
    # --- GPU最適化 & 省メモリ設定 (Largeモデル用) ---
    fp16=True,                       # 混合精度学習
    dataloader_num_workers=2,        # 読み込み並列化
    gradient_checkpointing=True,     # メモリ節約（計算時間は増えますがLargeには必須）
    
    # --- バッチサイズ調整 (極小バッチ + 蓄積) ---
    per_device_train_batch_size=2,   # 2に設定 (これで落ちるなら 1 にしてください)
    per_device_eval_batch_size=4,    # 評価用も小さめに
    gradient_accumulation_steps=8,   # 2×8 = 実質バッチサイズ16 で学習
    
    # --- 学習スケジュール ---
    num_train_epochs=3,              # 3周
    learning_rate=1e-5,              # Largeモデル用に少し学習率を下げる
    weight_decay=0.01,
    
    # --- 評価・保存 ---
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    
    # --- ログ ---
    logging_dir='./logs',
    logging_steps=50,
    
    seed=42,
    data_seed=42,
)

print("学習設定(BERT-Large用)の完了。")

学習設定(BERT-Large用)の完了。


c:\Users\blast\miniconda3\envs\torch-bert-env\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
# --- 7. Trainerの定義 ---
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 評価指標の計算関数
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Trainerの作成
trainer = Trainer(
    model=model,                        # 学習させるモデル
    args=training_args,                 # 先ほど設定したパラメータ
    train_dataset=tokenized_train_dataset, # 訓練データ
    eval_dataset=tokenized_eval_dataset,   # 評価データ
    tokenizer=tokenizer,                # トークナイザ
    compute_metrics=compute_metrics,    # 評価関数
)

print("Trainerの準備が完了しました。")

Trainerの準備が完了しました。


In [10]:
# --- 8. 学習の実行 ---
print("モデルのファインチューニングを開始します...")
trainer.train()

モデルのファインチューニングを開始します...


  0%|          | 0/792 [00:00<?, ?it/s]

c:\Users\blast\miniconda3\envs\torch-bert-env\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 0.6854, 'grad_norm': 6.296736717224121, 'learning_rate': 9.381313131313133e-06, 'epoch': 0.19}
{'loss': 0.6727, 'grad_norm': 5.7841057777404785, 'learning_rate': 8.750000000000001e-06, 'epoch': 0.38}
{'loss': 0.6659, 'grad_norm': 4.493304252624512, 'learning_rate': 8.11868686868687e-06, 'epoch': 0.57}
{'loss': 0.6279, 'grad_norm': 7.396188735961914, 'learning_rate': 7.487373737373738e-06, 'epoch': 0.76}
{'loss': 0.6096, 'grad_norm': 7.305905818939209, 'learning_rate': 6.856060606060606e-06, 'epoch': 0.95}


  0%|          | 0/264 [00:00<?, ?it/s]

...

c:\Users\blast\miniconda3\envs\torch-bert-env\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': 0.4507, 'grad_norm': 9.890894889831543, 'learning_rate': 3.0681818181818186e-06, 'epoch': 2.08}
{'loss': 0.3341, 'grad_norm': 11.25658130645752, 'learning_rate': 2.436868686868687e-06, 'epoch': 2.27}
{'loss': 0.3379, 'grad_norm': 8.794021606445312, 'learning_rate': 1.8055555555555557e-06, 'epoch': 2.46}
{'loss': 0.3442, 'grad_norm': 14.276654243469238, 'learning_rate': 1.1742424242424245e-06, 'epoch': 2.65}
{'loss': 0.3801, 'grad_norm': 12.516151428222656, 'learning_rate': 5.42929292929293e-07, 'epoch': 2.84}


  0%|          | 0/264 [00:00<?, ?it/s]

{'eval_loss': 0.6811943650245667, 'eval_accuracy': 0.6704545454545454, 'eval_f1': 0.6710775047258979, 'eval_precision': 0.6698113207547169, 'eval_recall': 0.6723484848484849, 'eval_runtime': 366.0296, 'eval_samples_per_second': 2.885, 'eval_steps_per_second': 0.721, 'epoch': 3.0}
{'train_runtime': 6609.712, 'train_samples_per_second': 1.917, 'train_steps_per_second': 0.12, 'train_loss': 0.5100911711201523, 'epoch': 3.0}


TrainOutput(global_step=792, training_loss=0.5100911711201523, metrics={'train_runtime': 6609.712, 'train_samples_per_second': 1.917, 'train_steps_per_second': 0.12, 'total_flos': 1.1809434236092416e+16, 'train_loss': 0.5100911711201523, 'epoch': 3.0})

In [11]:
# --- 8. 最終評価 ---
print("最終評価を行っています...")
eval_results = trainer.evaluate()

print("\n--- 最終評価結果 ---")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

最終評価を行っています...


  0%|          | 0/264 [00:00<?, ?it/s]


--- 最終評価結果 ---
eval_loss: 0.6090
eval_accuracy: 0.6951
eval_f1: 0.6793
eval_precision: 0.7164
eval_recall: 0.6458
eval_runtime: 357.5060
eval_samples_per_second: 2.9540
eval_steps_per_second: 0.7380
epoch: 3.0000
